In [1]:
# 1. Start fresh in the absolute base directory
%cd /content/
!rm -rf LedgerTune AgentTune AuditKIT

# 2. Clone the repositories
!git clone https://github.com/yashaswini450/LedgerTune.git
!git clone https://github.com/Lexsi-Labs/AgentTune.git
!git clone https://github.com/Lexsi-Labs/AuditKIT.git

# 3. Install the tools while strictly bypassing their dependency chains
!pip install -q --no-deps ./AgentTune ./AuditKIT

# 4. Move into your project directory and manually install the remaining ML libraries
%cd /content/LedgerTune
!pip install -q torch transformers datasets trl peft accelerate psutil

/content
Cloning into 'LedgerTune'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 37 (delta 4), reused 37 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 356.11 KiB | 15.48 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Cloning into 'AgentTune'...
remote: Enumerating objects: 985, done.
remote: Counting objects: 100% (985/985), done.
remote: Compressing objects: 100% (739/739), done.
remote: Total 985 (delta 254), reused 936 (delta 217), pack-reused 0 (from 0)
Receiving objects: 100% (985/985), 2.73 MiB | 30.41 MiB/s, done.
Resolving deltas: 100% (254/254), done.
Cloning into 'AuditKIT'...
remote: Enumerating objects: 1665, done.
remote: Counting objects: 100% (591/591), done.
remote: Compressing objects: 100% (305/305), done.
remote: Total 1665 (delta 377), reused 403 (delta 235), pack-reused 1074 (from 1)
Receiving objects: 100% (1665/1665), 3.56 MiB | 30.60 Mi

In [2]:
!pip install -q asteval bert-score bitsandbytes chromadb codebleu colorama compressed-tensors evaluate groq langchain-community langchain-text-splitters litellm lm-eval openenv pdfplumber psycopg2-binary python-docx python-pptx rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
agenttune 1.1.4 requires pdfplumber<0.11.10,>=0.10.0, but you have pdfplumber 0.11.10 which is incompatible.
agenttune 1.1.4 requires pillow<12,>=11.3.0, but you have pillow 12.3.0 which is incompatible.
agenttune 1.1.4 requires torch==2.11.0, but you have torch 2.14.1+cu130 which is 

In [3]:
!pip install -q "torchao>=0.17.0,<0.18.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
agenttune 1.1.4 requires pdfplumber<0.11.10,>=0.10.0, but you have pdfplumber 0.11.10 which is incompatible.
agenttune 1.1.4 requires pillow<12,>=11.3.0, but you have pillow 12.3.0 which is incompatible.
agenttune 1.1.4 requires torch==2.11.0, but you have torch 2.14.1+cu130 which is incompatible.
agenttune 1.1.4 requires trl==1.7.1, but you have trl 1.17.1 which is incompatible.


In [4]:
!python3 src/phase3_agenttune/train_real_agenttune.py \
    --base-model Qwen/Qwen2.5-1.5B-Instruct \
    --sft-epochs 3 \
    --grpo-steps 40

I0000 00:00:1794374042.237208    1889 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794374042.306874    1889 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794374044.493182    1889 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794374044.493818    1889 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

In [5]:
!git config user.name "Yashaswini"
!git config user.email "yashaswinikalpataru@gmail.com"
!git add data/auditkit_evaluation_report.json data/auditkit_evaluation_summary.md
!git commit -m "results: add live Qwen2.5-1.5B AgentTune SFT/GRPO rollout results via AuditKIT"
!git push

[main d838e8f] results: add live Qwen2.5-1.5B AgentTune SFT/GRPO rollout results via AuditKIT
 2 files changed, 1179 insertions(+), 895 deletions(-)
fatal: could not read Username for 'https://github.com': No such device or address


In [6]:
import shutil, zipfile
from pathlib import Path
from google.colab import files

root = Path("/content/LedgerTune") if Path("/content/LedgerTune").exists() else Path.cwd()

# 1. Create lightweight zip (< 5 MB) with all JSON/MD reports + training logs/configs for GitHub
github_zip = Path("/content/lexsi_github_results.zip")
with zipfile.ZipFile(github_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for rel in ["data/auditkit_evaluation_report.json", "data/auditkit_evaluation_summary.md"]:
        p = root / rel
        if p.exists():
            zf.write(p, arcname=rel)
    # Include all JSON/JSONL/MD files from checkpoints (adapter_config.json, trainer_state.json, trajectories.jsonl)
    # while skipping big .safetensors/.bin weight files so GitHub doesn't reject large files
    ckpt_root = root / "checkpoints"
    if ckpt_root.exists():
        for f in ckpt_root.rglob("*"):
            if f.is_file() and f.suffix in {".json", ".jsonl", ".md", ".txt", ".yaml"}:
                zf.write(f, arcname=str(f.relative_to(root)))

print(f"Created {github_zip} ({github_zip.stat().st_size / 1024:.1f} KB)")
files.download(str(github_zip))

Created /content/lexsi_github_results.zip (12658.5 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>